# Before your start:
- Read the README.md file
- Comment as much as you can and use the resources (README.md file)
- Happy learning!

In [1]:
# import numpy and pandas
import pandas as pd
import numpy as np
from scipy.stats import trim_mean, mode, skew, gaussian_kde, pearsonr, spearmanr, beta
from statsmodels.stats.weightstats import ztest as ztest

from scipy.stats import ttest_ind, norm, t
from scipy.stats import f_oneway
from scipy.stats import sem

# Challenge 1 - Exploring the Data

In this challenge, we will examine all salaries of employees of the City of Chicago. We will start by loading the dataset and examining its contents

In [2]:
# Run this code:
salaries = pd.read_csv('../data/Current_Employee_Names__Salaries__and_Position_Titles.csv')

Examine the `salaries` dataset using the `head` function below.

In [3]:
salaries.head(3)

,Name,Job Titles,Department,Full or Part-Time,Salary or Hourly,Typical Hours,Annual Salary,Hourly Rate
0,"AARON, JEFFERY M",SERGEANT,POLICE,F,Salary,NaN,101442.0,NaN
1,"AARON, KARINA",POLICE OFFICER (ASSIGNED AS DETECTIVE),POLICE,F,Salary,NaN,94122.0,NaN
2,"AARON, KIMBERLEI R",CHIEF CONTRACT EXPEDITER,GENERAL SERVICES,F,Salary,NaN,101592.0,NaN


In [4]:
salaries.describe

<bound method NDFrame.describe of                         Name                              Job Titles  \
0          AARON,  JEFFERY M                                SERGEANT   
1            AARON,  KARINA   POLICE OFFICER (ASSIGNED AS DETECTIVE)   
2        AARON,  KIMBERLEI R                CHIEF CONTRACT EXPEDITER   
3        ABAD JR,  VICENTE M                       CIVIL ENGINEER IV   
4          ABASCAL,  REECE E             TRAFFIC CONTROL AIDE-HOURLY   
...                      ...                                     ...   
33178  ZYLINSKA,  KATARZYNA                           POLICE OFFICER   
33179     ZYMANTAS,  LAURA C                          POLICE OFFICER   
33180      ZYMANTAS,  MARK E                          POLICE OFFICER   
33181    ZYRKOWSKI,  CARLO E                          POLICE OFFICER   
33182   ZYSKOWSKI,  DARIUSZ                  CHIEF DATA BASE ANALYST   

             Department Full or Part-Time Salary or Hourly  Typical Hours  \
0                POLICE 

# Challenge 2
This is a placeholder to make the AI corrector be able to find the correct exercise for feedback

# Challenge 3 - Constructing Confidence Intervals

We will test whether the hourly wage of all hourly workers is significantly different from $30/hr.

In the cell below, we will construct a 95% confidence interval for the mean hourly wage of all hourly workers. Is $30/hr within that interval?

The confidence interval is computed in SciPy using the `t.interval` function. You can read more about this function [here](https://docs.scipy.org/doc/scipy-0.14.0/reference/generated/scipy.stats.t.html).

To compute the confidence interval of the hourly wage, use the 0.95 for the confidence level, number of rows - 1 for degrees of freedom, the mean of the sample for the location parameter and the standard error for the scale. The standard error can be computed using [this](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.sem.html) function in SciPy.

In [9]:
# ---- Data cleaning ----
hourly_full = salaries[salaries['Salary or Hourly'] == 'Hourly'].copy()

# Check for missing values in 'Hourly Rate'
print(f"Missing hourly rates before drop: {hourly_full['Hourly Rate'].isna().sum()}")

# Drop rows where Hourly Rate is NaN
hourly_full = hourly_full.dropna(subset=['Hourly Rate'])

# Check for any unreasonable values (e.g., zero or negative)
print(f"Rows with Hourly Rate <= 0: {(hourly_full['Hourly Rate'] <= 0).sum()}")

# ---- Confidence interval computation ----
n_full = len(hourly_full)
mean_full = hourly_full['Hourly Rate'].mean()
se_full = sem(hourly_full['Hourly Rate'])
df_full = n_full - 1
confidence = 0.95

ci_lower_full, ci_upper_full = t.interval(confidence, df_full, loc=mean_full, scale=se_full)

print(f"\nNumber of hourly workers after cleaning: {n_full}")
print(f"Mean hourly wage: ${mean_full:.2f}")
print(f"Standard error: ${se_full:.2f}")
print(f"95% confidence interval: (${ci_lower_full:.2f}, ${ci_upper_full:.2f})")

if ci_lower_full <= 30 <= ci_upper_full:
    print(" $30/hr is within the confidence interval.")
else:
    print(" $30/hr is NOT within the confidence interval.")

Missing hourly rates before drop: 0
Rows with Hourly Rate <= 0: 0

Number of hourly workers after cleaning: 8022
Mean hourly wage: $32.79
Standard error: $0.14
95% confidence interval: ($32.52, $33.05)
 $30/hr is NOT within the confidence interval.


This is fine if we have thousands of worker data. But what if we have only 100 workers data?

Sample 100 workers and re-construct the 95% confidence interval. Is the interval wider of narrower? And why?
Do you still encapsulate the $30/hr mark in this case?

In [10]:
# Take a random sample of 100 from the data
hourly_sample = hourly_full.sample(n=100, random_state=42)  # fixed seed for reproducibility
n_samp = len(hourly_sample)
mean_samp = hourly_sample['Hourly Rate'].mean()
se_samp = sem(hourly_sample['Hourly Rate'])
df_samp = n_samp - 1

ci_lower_samp, ci_upper_samp = t.interval(confidence, df_samp, loc=mean_samp, scale=se_samp)

print(f"Sample size: {n_samp}")
print(f"Sample mean hourly wage: ${mean_samp:.2f}")
print(f"Standard error of sample: ${se_samp:.4f}")
print(f"95% confidence interval (sample): (${ci_lower_samp:.2f}, ${ci_upper_samp:.2f})")

# Compare widths
full_width = ci_upper_full - ci_lower_full
sample_width = ci_upper_samp - ci_lower_samp
print(f"\nWidth with full dataset: ${full_width:.2f}")
print(f"Width with sample of 100: ${sample_width:.2f}")

if sample_width > full_width:
    print(" The interval for the sample is WIDER because a smaller sample gives less precision.")
else:
    print(" The interval for the sample is NARROWER.")

if ci_lower_samp <= 30 <= ci_upper_samp:
    print(" $30/hr is within the sample confidence interval.")
else:
    print(" $30/hr is NOT within the sample confidence interval.")

Sample size: 100
Sample mean hourly wage: $33.46
Standard error of sample: $1.3242
95% confidence interval (sample): ($30.83, $36.08)

Width with full dataset: $0.53
Width with sample of 100: $5.25
 The interval for the sample is WIDER because a smaller sample gives less precision.
 $30/hr is NOT within the sample confidence interval.
